# Lifespan Predictor — Training Demo on NASA Turbofan Data

Predict **time to event** from histories in which some subjects have
not had the event yet. Here the subjects are aircraft engines from
NASA's C-MAPSS dataset, and the event is failure: the model predicts
each engine's remaining useful life.

The notebook runs end to end with no edits. It downloads the public
dataset, trains a model on SageMaker, shows its scores and calibration,
runs a batch transform and a real-time endpoint, and deletes what it made.

It runs on either Lifespan Predictor product, including the **free
tier**: FD001 has 100 training engines, within the free
tier's cap of 100 units. See [Which product?](https://github.com/kooditurva/lifespan-predictor-sdk#which-product)
for the two products and their algorithm ARNs.

**What you need**: an active AWS Marketplace subscription to either
product, and a SageMaker execution role.

## 1. Install the SDK

The client SDK is installed from its GitHub release tag. It is not
published to PyPI.

In [ ]:
# Other releases: https://github.com/kooditurva/lifespan-predictor-sdk/releases
!pip install -q "lifespan-predictor @ git+https://github.com/kooditurva/lifespan-predictor-sdk@v1.0.2"

## 2. Point at the algorithm you subscribed to

The ARN below is the free tier's, in `eu-west-1`. If you subscribed to
the other product or work in another Region, use the ARN listed under
[Which product?](https://github.com/kooditurva/lifespan-predictor-sdk#which-product), or find it in the AWS Marketplace
console: **Manage subscriptions**, select the product, choose
**Continue to configuration**, and pick your Region.

The execution role is the IAM role SageMaker assumes to run your
training jobs. The SDK reads it from `SAGEMAKER_ROLE_ARN`. Outside
SageMaker Studio, set that variable before starting Jupyter; in Studio,
the cell below sets it from `get_execution_role()`.

In [ ]:
import os

import matplotlib.pyplot as plt

from lifespan_predictor_sdk import (
    SurvivalModel,
    download_nasa_dataset,
    get_nasa_test_dataframe,
    get_nasa_train_dataframe,
)

ALGORITHM_ARN = 'arn:aws:sagemaker:eu-west-1:985815980388:algorithm/free-tier-lifespan-predictor-m-227e5356082e3b6ea8f8e5e080c1ef12'

if 'SAGEMAKER_ROLE_ARN' not in os.environ:
    from sagemaker import get_execution_role

    os.environ['SAGEMAKER_ROLE_ARN'] = get_execution_role()

### Check your setup

What the SDK will use. Set `SAGEMAKER_BUCKET` to keep the data and models
in a bucket of your own; otherwise SageMaker's default bucket for your
account and Region is used, and created if needed.

In [ ]:
from importlib.metadata import version

import boto3

region = boto3.Session().region_name
arn_region = ALGORITHM_ARN.split(':')[3]

print('SDK version:        ', version('lifespan-predictor'))
print('Region:             ', region)
print('AWS_PROFILE:        ', os.environ.get('AWS_PROFILE', '(not set)'))
print('SAGEMAKER_ROLE_ARN: ', os.environ['SAGEMAKER_ROLE_ARN'])
print('SAGEMAKER_BUCKET:   ', os.environ.get(
    'SAGEMAKER_BUCKET', "(not set: SageMaker's default bucket)"))
print('Algorithm ARN:      ', ALGORITHM_ARN)

if region != arn_region:
    print(f'Warning: the session is in {region}, but the algorithm is in '
          f'{arn_region}. Use the ARN for your Region.')

## 3. Load the NASA turbofan data

FD001 is one of the four C-MAPSS sub-datasets: simulated engines, each
observed once per flight cycle until it fails. The SDK downloads it and
returns it in the shape the algorithm expects:

| column | meaning |
|---|---|
| `unit` | the engine |
| `cycles` | the flight cycle within its history: `1, 2, 3, …` |
| `duration`, `event` | how long it ran, and whether it failed |
| everything else | operating settings and sensor readings |

The test engines' histories stop before failure. `test_labels_df` holds
their true remaining life, one row per engine, to score against.

In [ ]:
data_dir = 'data/nasa'
download_nasa_dataset(data_dir, datasets=[1])

train_df = get_nasa_train_dataframe(data_dir, datasets=[1])
test_df, test_labels_df = get_nasa_test_dataframe(data_dir, datasets=[1])

print(f"Training: {train_df['unit'].nunique()} engines, {len(train_df)} rows")
print(f"Test:     {test_df['unit'].nunique()} engines, {len(test_df)} rows")
train_df.head()

## 4. Train

`fit()` submits a SageMaker training job against the subscribed
algorithm. Nothing about the model runs locally.

- `model_type='deepsurv'` trains DeepSurv, a neural network that
  generalises the Cox proportional hazards model.
- `rolling_means` adds windowed averages of each sensor, which helps when
  signals drift before failure.
- `max_rul` caps the remaining life the model learns to predict.
- `calibration='isotonic'` calibrates the predicted probabilities on the
  validation split, which `val_split` holds out from the training engines.

Passing the test set and its labels makes the training job score the
model on them before it finishes.

In [ ]:
model = SurvivalModel({
    'algorithm_arn': ALGORITHM_ARN,
    'model_type': 'deepsurv',
    'instance_type': 'ml.m5.large',
    'num_nodes': [24],
    'epochs': 30,
    'rolling_means': [5, 10, 20, 30],
    'max_rul': 125,
    'calibration': 'isotonic',
})

model.fit(train_df, val_split=0.2, test_df=test_df, test_labels_df=test_labels_df)

## 5. Results

The training job's scores on the test engines:

- **C-index** measures ranking: given two engines, how often does the
  model put the one that failed first at higher risk? `0.5` is a coin flip.
- **RMSE** is the error in predicted remaining life, in cycles.
- The **calibration curve** compares predicted failure probabilities with
  how often engines actually failed by then. A calibrated model follows
  the diagonal.

In [ ]:
model.get_evaluation_result().report()

### Training history

Training and validation loss per epoch, and the learning rate.

In [ ]:
model.get_training_log().plot()
plt.show()

In [ ]:
model.print_training_job_summary()

## 6. Create a SageMaker model

`deploy()` turns the trained estimator into a SageMaker **Model**, which
is what batch transform runs against. Section 9 hosts it on a real-time
endpoint as well.

In [ ]:
model_name = model.deploy()
print('Created SageMaker model:', model_name)

## 7. Batch transform

`predict()` runs a batch transform job and returns, for each engine, the
probability of still running after each number of further cycles: one
column per engine, indexed by time. Each engine is predicted from the
end of its history.

In [ ]:
curves = model.predict(test_df)
units = sorted(test_df['unit'].unique())
print(f'{curves.shape[1]} survival curves, one per test engine')

ax = curves.iloc[:, :5].plot(drawstyle='steps-post', figsize=(8, 4))
ax.set_xlabel('further cycles')
ax.set_ylabel('survival probability')
ax.legend([f'engine {u}' for u in units[:5]])
plt.show()

## 8. Score it

`evaluate()` scores the deployed model's predictions against the true
remaining life in `test_labels_df`. It runs its own batch transform, and
should agree with the training job's scores in section 5. RMSE counts
only engines that actually failed: a censored one has no true time to
compare to.

In [ ]:
evaluation = model.evaluate(
    test_df,
    durations=test_labels_df['duration'].values,
    events=test_labels_df['event'].values,
)
print(f'C-index: {evaluation.c_index:.3f}')
print(f'RMSE:    {evaluation.rmse:.1f}')

evaluation.plot_calibration()
plt.show()

## 9. Real-time endpoint

`deploy(endpoint_instance_type=...)` hosts the model on an endpoint,
which answers in seconds instead of starting a job. `predict()` then
goes through it, with the same results as batch transform. An endpoint
is billed for as long as it exists, so the next section deletes it.

In [ ]:
model.deploy(endpoint_instance_type='ml.m5.large')

few = test_df[test_df['unit'].isin(units[:5])]
realtime = model.predict(few)
difference = float((realtime - curves.iloc[:, :5]).abs().max().max())
print(f'Largest difference from batch transform: {difference:.2e}')

## 10. Clean up

Delete the endpoint and the SageMaker model this notebook created, so
trying the product doesn't leave anything behind.

Training and transform jobs are billed for the time they ran and stop on
their own; there is nothing to delete for those.

In [ ]:
model.delete()
print('Deleted', model_name)

## Where to go next

- **Your own data**: rows per unit over time, with `unit`, `cycles`,
  `duration`, `event` and your features. On the free tier, train on at
  most 100 units.
- **Hyperparameter tuning**: put a range such as
  `ContinuousRange(1e-4, 1e-2, log=True)` where a value would go, and set
  `n_trials`. `fit()` then searches inside one training job.

Full documentation is in the SDK repository:
https://github.com/kooditurva/lifespan-predictor-sdk